In [1]:
# ================================
# 1. Librerías y sesión Spark
# ================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.stat import ChiSquareTest

spark = SparkSession.builder.appName("ChiSquareTest").getOrCreate()


In [2]:
# ================================
# 2. Cargar el dataset
# ================================
# Ojo: si tu carpeta es "merge_pyspark" con formato parquet, usa así:
merge_pyspark = spark.read.parquet("merge_pyspark")

merge_pyspark.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- Fecha: date (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: str

In [3]:
# ================================
# 3. Dividir en los dos subconjuntos
# ================================
conjunto1 = merge_pyspark.filter((col("Fecha") >= "2018-09-20") & (col("Fecha") <= "2019-12-31"))
conjunto2 = merge_pyspark.filter((col("Fecha") >= "2020-01-01") & (col("Fecha") <= "2020-09-22"))


In [4]:
# ================================
# 4. Función para aplicar Chi-cuadrado
# ================================
def chi_square_test(df, label_col="product_group_name", feature_col="sales_channel_id"):
    # Indexar etiqueta (product_group_name)
    indexer_label = StringIndexer(inputCol=label_col, outputCol="label_index")
    df = indexer_label.fit(df).transform(df)

    # Indexar feature (sales_channel_id)
    indexer_feat = StringIndexer(inputCol=feature_col, outputCol="feature_index")
    df = indexer_feat.fit(df).transform(df)

    # Armar vector de features
    assembler = VectorAssembler(inputCols=["feature_index"], outputCol="features")
    df = assembler.transform(df)

    # Test Chi-cuadrado
    result = ChiSquareTest.test(df, "features", "label_index").head()
    return {
        "Chi2": float(result.statistics[0]),
        "pValue": float(result.pValues[0]),
        "df": int(result.degreesOfFreedom[0])
    }


In [5]:
# ================================
# 5. Ejecutar en ambos conjuntos
# ================================
res1 = chi_square_test(conjunto1)
res2 = chi_square_test(conjunto2)



In [6]:
# ================================
# 6. Mostrar resultados comparativos con interpretación
# ================================
def interpretar(pvalue, alpha=0.05):
    if pvalue < alpha:
        return "Existe dependencia significativa"
    else:
        return "No se rechaza independencia"

results = [
    ("2018-2019", res1["Chi2"], res1["pValue"], res1["df"], interpretar(res1["pValue"])),
    ("2020", res2["Chi2"], res2["pValue"], res2["df"], interpretar(res2["pValue"]))
]

results_df = spark.createDataFrame(results, ["Conjunto", "Chi2", "pValue", "df", "Interpretación"])
results_df.show(truncate=False)




+---------+-----------------+------+---+--------------------------------+
|Conjunto |Chi2             |pValue|df |Interpretación                  |
+---------+-----------------+------+---+--------------------------------+
|2018-2019|910973.1869416058|0.0   |16 |Existe dependencia significativa|
|2020     |512042.4795792227|0.0   |18 |Existe dependencia significativa|
+---------+-----------------+------+---+--------------------------------+

